In [1]:
import duckdb

In [2]:
con = duckdb.connect('../change_tracker.db')

In [ ]:
drivers_average_score_data = con.sql(f"""                        
                        WITH latest_answers AS (
                            SELECT
                                q.id AS question_id,
                                d.drivers_name,
                                CAST(a.answers AS INTEGER) AS answers
                            FROM answers a
                            JOIN users u      ON u.id = a.user_id
                            JOIN questions q  ON q.id = a.questions_id
                            JOIN drivers d    ON d.id = q.drivers_id
                            QUALIFY ROW_NUMBER() OVER (
                                PARTITION BY q.id
                                ORDER BY a.modified_time DESC, a.id DESC
                            ) = 1
                        )
                        SELECT
                            drivers_name,
                            ROUND((SUM(answers) * 1.0 / COUNT(question_id)) / 7 * 100, 2) AS average_answer_percentage
                        FROM latest_answers
                        GROUP BY drivers_name;
                        """).fetchall()

drivers_question_data = con.sql(
                        f"""
                        WITH latest_answers AS (    
                            SELECT
                                q.id AS question_id,
                                d.drivers_name,
                                q.qcode,
                                CAST(a.answers AS INTEGER) AS answers
                            FROM answers a
                                JOIN users u      ON u.id = a.user_id
                                JOIN questions q  ON q.id = a.questions_id
                                JOIN drivers d    ON d.id = q.drivers_id
                                QUALIFY ROW_NUMBER() OVER (
                                PARTITION BY q.id
                                ORDER BY a.modified_time DESC, a.id DESC
                            ) = 1
                        )
                        SELECT
                            drivers_name,
                            qcode,
                            ROUND((SUM(answers) * 1.0 / COUNT(question_id)) / 7 * 100, 2) AS average_answer_percentage
                        FROM latest_answers
                        GROUP BY drivers_name, qcode ;
                        """
                        ).fetchall()

In [11]:
dict(drivers_average_score_data)

{'Business Performance': 42.86,
 'Team Leadership': 57.14,
 'Fear & Frustration': 57.14,
 'Accountability': 57.14,
 'Business Leadership': 57.14,
 'Benefits Realization': 45.24}

In [16]:
dict(drivers_average_score_data)['Business Performance']

42.86

In [ ]:
{'Business Performance': dict(drivers_average_score_data)['Business Performance']}

{'Business Performance': 42.86}

: 

In [10]:
{drivers_average_score_data[0][0]: drivers_average_score_data[0][1]}

{'Business Performance': 42.86}

In [4]:
con.sql('SELECT * FROM users').df()

,id,username,email
0,1,john_doe,john.doe@example.com
1,2,jane_smith,jane.smith@example.com
2,3,jane_smith,jane.smith@example.com
3,4,stephanie_song,shuet.yee.song@accenture.com
4,5,emily_wong,emily.a.wong@accenture.com
5,6,yap_shyne,shyne.yap@accenture.com
6,7,aisha_yusuf,aisha.yusuf@accenture.com
7,8,danial_mirza,danial.m.bin.madrawi@accenture.com


In [5]:
duckdb.read_csv('../data/ASTAR_responses.csv').df().columns

Index(['respondent id', 'username', 'Designation', 'TGPS Grouping',
       'enb|ldr_support_system', 'enb|ldr_time_resources', 'enb|conf_lv2_ldr',
       'sfb|current_change_mgmt', 'rsb|quick_remedial',
       'llb|leads_implementation', 'llb|performance_management',
       'llb|talents_utilised', 'llb|conf_lv5_ldr', 'llb|recognised_rewarded',
       'llb|role_clarity', 'llb|accountable', 'llb|objectives_outcomes',
       'eeb|fear', 'eeb|distress', 'eeb|anger'],
      dtype='object')

In [ ]:
# for username, email in con.sql("select lower(split_part(username, '@', 1)) as username, lower(username) as email from '../data/ASTAR_responses.csv'").fetchall():
#     con.sql(f"INSERT INTO users (id, username, email) VALUES (NEXTVAL('users_id_seq'),'{username}', '{email}')")

In [15]:
con.sql(f"select * from users").df()

,id,username,email
0,1,john_doe,john.doe@example.com
1,2,jane_smith,jane.smith@example.com
2,3,jane_smith,jane.smith@example.com
3,4,stephanie_song,shuet.yee.song@accenture.com
4,5,emily_wong,emily.a.wong@accenture.com
...,...,...,...
364,365,jerald_chua,jerald_chua@hq.a-star.edu.sg
365,366,chua_yirong,chua_yirong@hq.a-star.edu.sg
366,367,koid_wei_ling,koid_wei_ling@hq.a-star.edu.sg
367,368,spring_kan,spring_kan@hq.a-star.edu.sg


In [ ]:
for i in con.sql("select lower(split_part(username, '@', 1)) as username from '../data/ASTAR_responses.csv'").fetchnumpy()['username'].tolist():
    con.sql(f"INSERT INTO users (username) VALUES ('{i}')")

kenneth_kuan
jolin_chua
kenny_soh
connie_chan
witri_ramli
grace_lai
evita_ang
weegn
kathleen_chong
joyce_tay
dhanaraj_kumaravel
clarissa_choo
tansmj
celestina_foo
liew_wee_kiat
lim_kai_boon
chiam_mei_hui
lea_ling_nui
christy_tan
peggy_low
pauline_tan
wee_min_zhi
angie_ng
lovelle_lam
eileen_lee
garry_chung
ho_lai_ping
phurjt
jolene_cheng
ng_jing_ying
moh_yun_fen
susan_lau
ng_gim_yeow
ashley_tan
joseph_ong
bala_narayanan
chua_lay_hong
cheng_wai_man
azlinah_ahmad_said
edwin_lim
felicia_goh
ng_heng_tio
pang_shihui
soon_mei_yi
ram_sakthi
hendri_jaya
tay_chay_wah
lim_bee_sian
elissa_lim
tan_hsiao_yuen
gan_wei_xuan
jerica_tan
jasmine_yuen
yeo_li_tong
leng_chee_weng
lim_gek_lian
cw-wong
noor_nisha
dorothy_yeo
ong_hui_yong
serene_giam
yeo_yew_peng
nariyah_ismail
han_yong_ping
kristine_kow
jerome_tang
athar_ali_buang
tanhz
rachel_teo_ngi_ching
chanel_ng
kelvin_tan
jessica_tan
tan_jie_shi
wang_junfeng
julie_loke
jolene_cheah
sherlyn_wong
pouw_lih_woei
lohhpc
pauline_chua
kelly_tan
amirah_amran
md

In [25]:
con.sql(""" 
        
        with answers as (
        unpivot '../data/ASTAR_responses.csv'
            on 
            "enb|ldr_support_system", "enb|ldr_time_resources", "enb|conf_lv2_ldr",
            "sfb|current_change_mgmt", "rsb|quick_remedial",
            "llb|leads_implementation", "llb|performance_management",
            "llb|talents_utilised", "llb|conf_lv5_ldr", "llb|recognised_rewarded",
            "llb|role_clarity", "llb|accountable", "llb|objectives_outcomes",
            "eeb|fear", "eeb|distress", "eeb|anger"
        into NAME drivers VALUE scores)
        select * from answers
        """)

┌───────────────┬───────────────────────────────┬─────────────┬──────────────────────┬────────────────────────────┬────────┐
│ respondent id │           username            │ Designation │    TGPS Grouping     │          drivers           │ scores │
│     int64     │            varchar            │   varchar   │       varchar        │          varchar           │ int64  │
├───────────────┼───────────────────────────────┼─────────────┼──────────────────────┼────────────────────────────┼────────┤
│      23891377 │ KENNETH_KUAN@HQ.A-STAR.EDU.SG │ Manager     │ Corporate Group - HR │ enb|ldr_support_system     │      4 │
│      23891377 │ KENNETH_KUAN@HQ.A-STAR.EDU.SG │ Manager     │ Corporate Group - HR │ enb|ldr_time_resources     │      6 │
│      23891377 │ KENNETH_KUAN@HQ.A-STAR.EDU.SG │ Manager     │ Corporate Group - HR │ enb|conf_lv2_ldr           │      6 │
│      23891377 │ KENNETH_KUAN@HQ.A-STAR.EDU.SG │ Manager     │ Corporate Group - HR │ sfb|current_change_mgmt    │      5 │


In [38]:
answers = con.sql(""" 
        
        with answers_temp as (
        unpivot '../data/ASTAR_responses.csv'
            on 
            "enb|ldr_support_system", "enb|ldr_time_resources", "enb|conf_lv2_ldr",
            "sfb|current_change_mgmt", "rsb|quick_remedial",
            "llb|leads_implementation", "llb|performance_management",
            "llb|talents_utilised", "llb|conf_lv5_ldr", "llb|recognised_rewarded",
            "llb|role_clarity", "llb|accountable", "llb|objectives_outcomes",
            "eeb|fear", "eeb|distress", "eeb|anger"
        into NAME drivers VALUE scores)
        select u.id user_id, q.id questions_id, atemp.scores  from answers_temp atemp
            inner join users u on u.username = lower(split_part(atemp.username, '@', 1))
            inner join questions q on q.qcode = atemp.drivers
        
        ;

        """).fetchall()

In [ ]:
# for user_id, questions_id, scores in answers:
    # con.sql(f"INSERT INTO answers (user_id, questions_id, answers) VALUES ({user_id}, {questions_id}, {scores})")